# F-MI-N-SUPP-ARC Multifrequency Sensor-Noise Arc Geometry Baseline

Direct frequency-domain multifrequency sensor-noise support-recovery simulations with 3 microphones and 10 candidate sources restricted to arc sectors. The sector starts at 0 degrees and spans 10, 25, or 45 degrees. Grouped-method entry points are run without grouping structure.


In [ ]:
from pathlib import Path
import os
import sys


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "src" / "cs_priors").exists():
            return path
    raise RuntimeError("Could not find repository root containing src/cs_priors")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/cs_priors_matplotlib")

from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from cs_priors.plotting.plot_geometry import plot_geometry_on_ax
from cs_priors.simulation.mixing_model import quick_frequency_sim
from cs_priors.solvers.freq_lasso import frequency_lasso_solve
from cs_priors.solvers.freq_group_lasso import frequency_group_lasso_solve
from cs_priors.solvers.freq_sparse_prior import sparse_prior_solve


In [ ]:
TAG = "F-MI-N-SUPP-ARC"
FIGURE_DIR = REPO_ROOT / "results" / "benchmarks" / TAG
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig, stem: str):
    fig.savefig(FIGURE_DIR / f"{stem}.pdf", dpi=300, bbox_inches="tight")


SECTOR_DEG = np.array([10.0, 25.0, 45.0])
SNR_DB = np.array([10, 15, 20, 30, 40, 50, 60])
SEEDS = np.arange(5)

NUM_MICS = 3
NUM_SOURCES = 10
NUM_ACTIVE = 2
MIN_FREQ_HZ = 100.0
SAMPLING_RATE = 2000.0
DURATION = 0.05
COMPONENT_AMPLITUDE = 1.0

MIC_RADIUS = 0.3
SOURCE_DISTANCE = 1.5
ANGLE_START = 0.0

LASSO_ALPHA = 1e-4
GROUP_LASSO_ALPHA = 1e-4
LASSO_MAX_ITER = 5_000
GROUP_LASSO_MAX_ITER = 500
PLOT_FLOOR = 1e-16
GROUPING = "none"

METHOD_ORDER = ["r-LASSO", "Group LASSO", "Sparse Prior", "X_pinv"]
METHOD_COLORS = {
    "X_pinv": "tab:blue",
    "r-LASSO": "tab:orange",
    "Group LASSO": "tab:green",
    "Sparse Prior": "tab:red",
}
METHOD_LINESTYLES = {
    "X_pinv": ":",
    "r-LASSO": "-",
    "Group LASSO": "-",
    "Sparse Prior": "-",
}


In [ ]:
def sector_rad(sector_deg: float) -> float:
    return float(np.deg2rad(sector_deg))


def make_simulation(sector_deg: float, snr_db: float, seed: int):
    span = sector_rad(sector_deg)
    return quick_frequency_sim(
        num_mics=NUM_MICS,
        num_sources=NUM_SOURCES,
        num_active=NUM_ACTIVE,
        seed=int(seed),
        sampling_rate=SAMPLING_RATE,
        duration=DURATION,
        source_distance=SOURCE_DISTANCE,
        mic_radius=MIC_RADIUS,
        array_type="arc",
        mic_angle_start=ANGLE_START,
        mic_angle_span=span,
        source_angle_start=ANGLE_START,
        source_angle_span=span,
        component_amplitude=COMPONENT_AMPLITUDE,
        magnitude_jitter=0.0,
        min_freq_hz=MIN_FREQ_HZ,
        single_frequency_hz=None,
        sensor_snr_db=float(snr_db),
        model_snr_db=None,
        inverse_method="mp",
    )


def solve_methods(sim) -> dict[str, np.ndarray]:
    X_pinv = sim.X_pinv
    return {
        "X_pinv": X_pinv,
        "r-LASSO": frequency_lasso_solve(
            sim.Y, sim.A, alpha=LASSO_ALPHA, max_iter=LASSO_MAX_ITER, X_start=X_pinv
        ),
        "Group LASSO": frequency_group_lasso_solve(
            sim.Y,
            sim.A,
            alpha=GROUP_LASSO_ALPHA,
            grouping=GROUPING,
            max_iter=GROUP_LASSO_MAX_ITER,
            X_start=X_pinv,
        ),
        "Sparse Prior": sparse_prior_solve(X_pinv, sim.A, grouping=GROUPING),
    }


def mean_squared_difference(X_hat: np.ndarray, X_true: np.ndarray) -> float:
    return float(np.mean(np.abs(X_hat - X_true) ** 2))


def mse_to_db(values):
    return 10.0 * np.log10(np.maximum(np.asarray(values, dtype=float), PLOT_FLOOR))


def assert_simulation_contract(sim):
    assert sim.A.shape == (NUM_MICS, NUM_SOURCES, 45)
    assert np.all(sim.freqs >= MIN_FREQ_HZ)
    assert len(sim.active_indices) == NUM_ACTIVE
    assert np.allclose(sim.delta, 0.0)
    assert np.linalg.norm(sim.eta) > 0.0


def run_case(sector_deg: float, snr_db: float, seed: int) -> list[dict]:
    sim = make_simulation(sector_deg, snr_db, seed)
    assert_simulation_contract(sim)

    rows = []
    for method, X_hat in solve_methods(sim).items():
        rows.append(
            {
                "tag": TAG,
                "sector_deg": float(sector_deg),
                "sensor_snr_db": float(snr_db),
                "seed": int(seed),
                "method": method,
                "mean_squared_difference": mean_squared_difference(X_hat, sim.X),
            }
        )
    return rows


In [ ]:
fig, axes = plt.subplots(1, len(SECTOR_DEG), figsize=(12.0, 4.0), sharex=False, sharey=False)
axes = np.atleast_1d(axes)

for ax, sector_deg in zip(axes, SECTOR_DEG):
    sim = make_simulation(sector_deg=float(sector_deg), snr_db=60.0, seed=0)
    assert_simulation_contract(sim)
    plot_geometry_on_ax(ax, sim.mics, sim.sources, pad_factor=0.2)
    for k, idx in enumerate(sim.active_indices):
        sx, sy = sim.sources[idx].get_position()
        ax.scatter(
            sx,
            sy,
            s=80,
            marker="^",
            color="gold",
            edgecolor="k",
            label="Active sources" if k == 0 else None,
        )
    ax.set_title(f"Sector = {sector_deg:g} deg")
    ax.legend(frameon=False, fontsize=8)

fig.suptitle(f"Arc geometries: M={NUM_MICS}, S={NUM_SOURCES}, start angle=0 deg", y=1.02)
fig.tight_layout()
save_figure(fig, "simulation_geometry_by_sector")
plt.show()


In [ ]:
rows = []
cases = product(SECTOR_DEG, SNR_DB, SEEDS)
total = len(SECTOR_DEG) * len(SNR_DB) * len(SEEDS)

for sector_deg, snr_db, seed in tqdm(cases, total=total, desc="Running simulations"):
    rows.extend(run_case(float(sector_deg), float(snr_db), int(seed)))

results_df = pd.DataFrame(rows)
results_df["method"] = pd.Categorical(results_df["method"], categories=METHOD_ORDER, ordered=True)
results_df = results_df.sort_values(["sector_deg", "sensor_snr_db", "seed", "method"]).reset_index(drop=True)
results_df.head()


In [ ]:
summary_df = (
    results_df
    .groupby(["sector_deg", "sensor_snr_db", "method"], observed=False)
    .agg(
        median_mean_squared_difference=("mean_squared_difference", "median"),
        q25_mean_squared_difference=("mean_squared_difference", lambda x: x.quantile(0.25)),
        q75_mean_squared_difference=("mean_squared_difference", lambda x: x.quantile(0.75)),
    )
    .reset_index()
)

summary_df["median_mean_squared_difference_db"] = mse_to_db(summary_df["median_mean_squared_difference"])
summary_df["q25_mean_squared_difference_db"] = mse_to_db(summary_df["q25_mean_squared_difference"])
summary_df["q75_mean_squared_difference_db"] = mse_to_db(summary_df["q75_mean_squared_difference"])
summary_df.head()


In [ ]:
fig, axes = plt.subplots(1, len(SECTOR_DEG), figsize=(13.0, 4.2), sharey=True)
axes = np.atleast_1d(axes)

for ax, sector_deg in zip(axes, SECTOR_DEG):
    plot_df = summary_df[summary_df["sector_deg"] == float(sector_deg)]
    for method in METHOD_ORDER:
        method_df = plot_df[plot_df["method"] == method].sort_values("sensor_snr_db")
        x = method_df["sensor_snr_db"].to_numpy(dtype=float)
        median = method_df["median_mean_squared_difference_db"].to_numpy(dtype=float)
        q25 = method_df["q25_mean_squared_difference_db"].to_numpy(dtype=float)
        q75 = method_df["q75_mean_squared_difference_db"].to_numpy(dtype=float)

        ax.plot(
            x,
            median,
            marker="o",
            label=method,
            color=METHOD_COLORS[method],
            linestyle=METHOD_LINESTYLES[method],
        )
        ax.fill_between(x, q25, q75, color=METHOD_COLORS[method], alpha=0.18, linewidth=0)

    ax.set_title(f"Sector = {sector_deg:g} deg")
    ax.set_xticks(SNR_DB)
    ax.set_xlabel("Sensor SNR (dB)")
    ax.grid(True, linewidth=0.5, alpha=0.35)

axes[0].set_ylabel("Mean squared difference (dB)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(METHOD_ORDER), frameon=False)
fig.suptitle(f"Arc-sector coefficient MSE vs. sensor SNR (M={NUM_MICS}, S={NUM_SOURCES})", y=1.02)
fig.tight_layout(rect=(0, 0.10, 1, 1))

save_figure(fig, "mean_squared_difference_db_vs_sensor_snr_by_sector")
plt.show()


## Interpretation

This notebook compares coefficient reconstruction error across local arc-sector geometries. Both microphones and candidate sources start at 0 degrees and span the configured sector width. The grouped-method entry points use `grouping="none"`, so cross-frequency structure is intentionally not used.
